# TRIP Data Integration Script

This script merges annotated barcodes tables with feature tables (such as histone marks, speckle associations, U2AF2 binding, etc.) based on barcode identity. Each feature set is provided in a separate file per experiment. Barcodes are used to align rows between different data sources.

---

## Example of Directory Structure

The script expects the following directory layout (relative to the project root):

<pre>

project_root/
├── annotated_tables/
│   ├── EXP1.annotated_tables.txt
│   └── EXP2.annotated_tables.txt
│
├── combined/
│   ├── EXP1.combined_histone.txt
│   ├── EXP1.combined_speckle.txt
│   ├── EXP1.combined_stranded.txt
│   ├── EXP2.combined_histone.txt
│   ├── EXP2.combined_speckle.txt
│   └── EXP2.combined_stranded.txt
│
├── barcodes_order/                     ← row order for feature-files
│   ├── EXP1.hg38_barcodes.txt          # for histone
│   ├── EXP1.hg19_barcodes.txt          # for speckle / u2af2
│   └── EXP1.stranded_barcodes.txt      # for stranded 
│
├── barcodes_bed/          
│   ├── EXP1.bed
│   └── EXP2.bed
│
├── columns_names/
│   ├── histone_names.txt
│   ├── speckle_names.txt
│   └── stranded_names.txt
│
├── names_all_clean.txt       # list of experiment names
└── output/
    ├── EXP1.alldata.tab
    └── EXP2.alldata.tab

</pre>

Directory names are specified when specifying paths to them (this is an example)


---

## File Format Requirements

### 1. `annotated_tables/{exp}.annotated_tables.txt`
- Tab-separated values (TSV) with header.
- First column: barcode sequence (used as index).
- Other columns: annotation features (e.g., splicing efficiency, expression).

### 2. `combined/*{exp}*combined*{ftype}*.txt`
- Tab-separated values (TSV).
- No header row.
- Each row corresponds to a barcode.
- Column names provided separately (see below).

### 3. `barcodes_order/*.txt`
- One barcode per line.
- Order must match the row order in the corresponding `.combined` file.
- Barcode file suffix depends on feature type. For example:
  - `histone` → `.hg38_barcodes.txt`
  - `speckle`, `u2af2` → `.hg19_barcodes.txt`
  - `stranded` → `.stranded_barcodes.txt`

### 4. `columns_names/{ftype}_names.txt`
- One feature (column) name per line.
- Order must match the columns in the corresponding `.combined` file.

### 5. `names_all_clean.txt`
- One experiment name per line.

### 6. Selected Columns (optional)
- You may define a list of selected columns to keep from each feature type.
- Each line should contain one column name.
- These should be listed in a file in a directory such as:

## File Naming Summary

| Data Type        | File Pattern                                      |
|------------------|---------------------------------------------------|
| Annotated table  | `{experiment_name}.annotated_tables.txt`          |
| Feature table    | `*{experiment_name}*combined*{feature_type}*.txt` |
| Feature names    | `columns_names/{feature_type}_names.txt`          |
| Barcodes         | `*{experiment_name}*{suffix}`                     |


## Feature Type Configuration

You must define the list of features you want to process in the script using:

```python
feature_types = ['histone', 'speckle', 'u2af2', 'stranded']
```
---

# Input

#### === PATH SETUP ===

In [ ]:
from settings.utils import resolve_path, PROJECT_ROOT

In [ ]:
# for single_combined (5ss_noP, CPA_noP)
# The DATA/single_combined/ folder is built by DATA/single_combined/prepare_dataset.ipynb,
# which glues the feature matrices from the per-track files itself. Only histone and speckle
# exist here, so feature_types below must not list stranded or u2af2.
path_to_dir_with_colnames = resolve_path('DATA/single_combined/columns_names')
path_to_dir_with_annotated_tables = resolve_path('DATA/single_combined/annotated_tables')
path_to_dir_with_barcodes_order = resolve_path('DATA/single_combined/barcodes_order')
path_to_dir_with_bed_files = resolve_path('DATA/single_combined/barcodes_bed')
path_to_dir_with_combined_features = resolve_path('DATA/single_combined/combined')
path_to_dir_with_selected_columns = resolve_path('DATA/single_combined/selected_columns')
path_to_file_with_experiments_names = resolve_path('DATA/single_combined/names_all_clean.txt')

# do not change unless necessary
output_dir = resolve_path('tables_for_training/single_combined')

In [ ]:
# # for intron_exp (8 experiments, one per intron)
# # The DATA/intron_exp/ folder is built by DATA/intron_exp/prepare_dataset.ipynb. Only histone
# # and speckle exist here, so feature_types must not list stranded or u2af2.
# path_to_dir_with_colnames = resolve_path('DATA/intron_exp/columns_names')
# path_to_dir_with_annotated_tables = resolve_path('DATA/intron_exp/annotated_tables')
# path_to_dir_with_barcodes_order = resolve_path('DATA/intron_exp/barcodes_order')
# path_to_dir_with_bed_files = resolve_path('DATA/intron_exp/barcodes_bed')
# path_to_dir_with_combined_features = resolve_path('DATA/intron_exp/combined')
# path_to_dir_with_selected_columns = resolve_path('DATA/intron_exp/selected_columns')
# # path_to_file_with_experiments_names = resolve_path('DATA/intron_exp/names_all_clean_test.txt')
# path_to_file_with_experiments_names = resolve_path('DATA/intron_exp/names_all_clean.txt')

# # do not change unless necessary
# output_dir = resolve_path('tables_for_training/intron_exp')

In [ ]:
# for intron_splicing (8 experiments, one per intron)
# The DATA/intron_splicing/ folder is built by DATA/intron_splicing/prepare_dataset.ipynb.
# The target exp.log10 is a splicing fraction in [0, 1], not an expression value.
# Only histone and speckle exist here, so feature_types must not list stranded or u2af2.
# path_to_dir_with_colnames = resolve_path('DATA/intron_splicing/columns_names')
# path_to_dir_with_annotated_tables = resolve_path('DATA/intron_splicing/annotated_tables')
# path_to_dir_with_barcodes_order = resolve_path('DATA/intron_splicing/barcodes_order')
# path_to_dir_with_bed_files = resolve_path('DATA/intron_splicing/barcodes_bed')
# path_to_dir_with_combined_features = resolve_path('DATA/intron_splicing/combined')
# path_to_dir_with_selected_columns = resolve_path('DATA/intron_splicing/selected_columns')
# path_to_file_with_experiments_names = resolve_path('DATA/intron_splicing/names_all_clean_test.txt')
# path_to_file_with_experiments_names = resolve_path('DATA/intron_splicing/names_all_clean.txt')

# do not change unless necessary
# output_dir = resolve_path('tables_for_training/intron_splicing')

#### === BARCODE FILE PICKER FUNCTION ===

In [ ]:
def get_barcode_file(exp, ftype):
    if ftype == 'histone':
        suffix = '.hg38_barcodes.txt'
    elif ftype in ['speckle', 'u2af2']:
        suffix = '.hg19_barcodes.txt'
    elif ftype == 'stranded':
        suffix = '.stranded_barcodes.txt'
    else:
        return None
    pattern = os.path.join(path_to_dir_with_barcodes_order, f'*{exp}*{suffix}')
    matches = glob.glob(pattern)
    return matches[0] if matches else None

#### === FEATURE TYPES (AUTODETECTABLE) ===

In [ ]:
feature_types = ['histone', 'speckle']   # intron_exp / intron_splicing / single_combined have only these two

# Start

In [ ]:
import os
import pandas as pd
import glob
os.makedirs(output_dir, exist_ok=True)

# === SELECT COLUMNS ===
files = [f for f in os.listdir(path_to_dir_with_selected_columns) if os.path.isfile(os.path.join(path_to_dir_with_selected_columns, f)) and not f.startswith(".")]
selected_columns = []
for f in files:
    with open(f'{path_to_dir_with_selected_columns}/{f}', 'r') as file_:
        for j in file_.readlines():
            selected_columns.append(j.strip())


# === READ EXPERIMENT NAMES ===
with open(path_to_file_with_experiments_names, 'r') as f:
    experiment_names = [line.strip() for line in f if line.strip()]

# === MAIN PROCESSING LOOP ===
for exp in experiment_names:
    print(f'\n[Processing] {exp}')
    dfs = []  # list of dataframes to merge
    barcodes_lists = []  # track barcode lists for later filtering

    # --- Load annotated table ---
    annotated_path = os.path.join(path_to_dir_with_annotated_tables, f'{exp}.annotated_tables.txt') # check
    if not os.path.exists(annotated_path):
        print(f'  [!] Annotated table not found for {exp}')
        continue
    annotated_df = pd.read_csv(annotated_path, sep='\t', index_col=0)
    dfs.append(annotated_df)
    barcodes_lists.append(set(annotated_df.index))

    # --- Load bed file ---
    bed_path = os.path.join(path_to_dir_with_bed_files, f'{exp}*.bed') # check - hg38
    bed_path = glob.glob(bed_path)[0]
    if not os.path.exists(bed_path):
        print(f'  [!] Bed file not found for {exp}')
        continue
    bed_file = pd.read_csv(bed_path,
                       sep='\t', header=None, index_col=3)
    bed_file.index.name = None
    bed_file.columns = ['chr', 'start', 'end', 'score', 'strand']

    # --- Load all feature tables ---
    for ftype in feature_types:
        feature_pattern = os.path.join(path_to_dir_with_combined_features, f'*{exp}*combined*{ftype}*.txt') # check
        feature_files = glob.glob(feature_pattern)

        if len(feature_files) == 1:
            feature_file = feature_files[0]
        elif len(feature_files) == 0:
            print(f"  [!] No combined feature file for {ftype}")
            continue
        else:
            print(f"  [!] Multiple feature files found for {ftype}: {feature_files}")
            continue

        colnames_path = os.path.join(path_to_dir_with_colnames, f'{ftype}_names.txt') # check
        if not os.path.exists(colnames_path):
            print(f'  [!] Column names file missing for {ftype}')
            continue

        # Load feature values
        with open(colnames_path) as f:
            colnames = [line.strip() for line in f if line.strip()]
        feature_df = pd.read_csv(feature_file, sep='\t', header=None, names=colnames)

        # Load barcodes
        barcode_file = get_barcode_file(exp, ftype)
        if not barcode_file or not os.path.exists(barcode_file):
            print(f'  [!] Barcode file missing for {ftype}')
            continue
        with open(barcode_file) as f:
            barcodes = [line.strip() for line in f if line.strip()]

        if len(barcodes) != len(feature_df):
            print(f'  [!] Barcode count does not match feature rows for {ftype}')
            continue

        feature_df.index = barcodes
        barcodes_lists.append(set(barcodes))
        dfs.append(feature_df)

        print(f'  [+] {ftype} loaded ({feature_df.shape[0]} rows, {feature_df.shape[1]} features)')

    # --- Find common barcodes across all ---
    common_barcodes = set.intersection(*barcodes_lists)
    if not common_barcodes:
        print(f'  [!] No common barcodes found across data for {exp}')
        continue

    # --- Filter and align data ---
    dfs_filtered = [df.loc[df.index.intersection(common_barcodes)].sort_index() for df in dfs]
    final_df = pd.concat(dfs_filtered, axis=1)
    final_df = final_df[selected_columns].copy()
    final_df = pd.merge(bed_file, final_df, left_index=True, right_index=True, how='inner')


    # --- Save result ---
    output_path = os.path.join(output_dir, f'{exp}.alldata.tab')
    final_df.to_csv(output_path, sep='\t', index=True)
    print(f'  [✔] Saved merged data to {output_path}, \n \t ({final_df.shape[0]} rows, {final_df.shape[1]} features)')
print('[✔] Processing completed [✔]')